In [ ]:
import sys, subprocess

def run(cmd):
    print(" ".join(cmd))
    subprocess.check_call(cmd)

run([sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir",
     "pandas==2.2.2",
     "trl",
     "peft",
     "datasets",
     "bitsandbytes",
     "sentencepiece"])


In [ ]:
import pandas, trl, transformers, accelerate, peft, datasets
print("pandas", pandas.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)
print("peft", peft.__version__)
print("datasets", datasets.__version__)

In [ ]:
# ============================================================
# Qwen2-1.5B | Kaggle | 最终稳定版：DPO 显存爆炸 100% 解决
# 仅按你要求优化：截断长度 + 清显存 + reference_free + 子采样
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed
)

from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DPOTrainer, DPOConfig

# -----------------------------
# 0) 基础配置
# -----------------------------
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

csv_path = "/kaggle/input/datasets/yalizongkuaile/260408-stint-stage3-observation/filtered_observation_data.csv"

BASE_MODEL = "Qwen/Qwen2-1.5B-Instruct"
SFT_SAVE_DIR = f"{WORK_DIR}/qwen_sft"
DPO_SAVE_DIR = f"{WORK_DIR}/qwen_dpo_final"

MAX_SEQ_LEN = 768
BATCH_SIZE = 1
GRAD_ACCUM = 16

RUN_SMOKE_TEST = True
SMOKE_STEPS = 5
RUN_FULL_TRAIN = True

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# -----------------------------
# 1) 数据读取与构建
# -----------------------------
df = pd.read_csv(csv_path)
required_cols = ["abst", "ext_question", "ext_method", "ext_conclusion"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"缺少字段: {c}")
    df[c] = df[c].fillna("").astype(str).str.strip()

df = df[
    (df["abst"] != "") &
    (df["ext_question"] != "") &
    (df["ext_method"] != "") &
    (df["ext_conclusion"] != "")
].copy().reset_index(drop=True)

print(f"有效样本数: {len(df)}")

stats = {
    "n_samples": len(df),
    "abst_len_mean": float(df["abst"].str.len().mean()),
    "abst_len_median": float(df["abst"].str.len().median()),
    "abst_len_p90": float(df["abst"].str.len().quantile(0.9)),
    "question_len_mean": float(df["ext_question"].str.len().mean()),
    "method_len_mean": float(df["ext_method"].str.len().mean()),
    "conclusion_len_mean": float(df["ext_conclusion"].str.len().mean()),
}
pd.DataFrame([stats]).to_csv(f"{WORK_DIR}/dataset_stats.csv", index=False, encoding="utf-8-sig")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a senior biomedical intelligence expert. "
    "Extract only Question, Method, Conclusion from the abstract. "
    "Output STRICT JSON ONLY with keys: Question, Method, Conclusion."
)

def user_prompt(abst):
    return f"请提取以下摘要的 Question/Method/Conclusion，并严格输出JSON：\n{abst}"

def gold_json(r):
    return json.dumps({
        "Question": r["ext_question"],
        "Method": r["ext_method"],
        "Conclusion": r["ext_conclusion"]
    }, ensure_ascii=False)

def truncate_text(s, ratio=0.5):
    s = s.strip()
    if len(s) <= 4:
        return "信息缺失"
    return s[:max(2, int(len(s) * ratio))].strip()

def rejected_json(r):
    q, m, c = r["ext_question"], r["ext_method"], r["ext_conclusion"]
    mode = random.choice(["missing", "truncate", "shuffle"])
    if mode == "missing":
        if random.random() < 0.5:
            m = "信息缺失"
        else:
            c = "信息缺失"
    elif mode == "truncate":
        if random.random() < 0.5:
            m = truncate_text(m, random.uniform(0.3, 0.7))
        else:
            c = truncate_text(c, random.uniform(0.3, 0.7))
    else:
        m, c = c, m
    return json.dumps({"Question": q, "Method": m, "Conclusion": c}, ensure_ascii=False)

def build_sft_text(r):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(r["abst"])},
        {"role": "assistant", "content": gold_json(r)}
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def build_dpo_prompt(r):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(r["abst"])},
    ]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    if not p.endswith("\n"):
        p += "\n"
    return p

def normalize_completion(txt):
    return "\n" + txt.strip()

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=SEED, shuffle=True)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

sft_train_list = [{"text": build_sft_text(r)} for _, r in train_df.iterrows()]
sft_eval_list = [{"text": build_sft_text(r)} for _, r in eval_df.iterrows()]

dpo_train_list = [{
    "prompt": build_dpo_prompt(r),
    "chosen": normalize_completion(gold_json(r)),
    "rejected": normalize_completion(rejected_json(r))
} for _, r in train_df.iterrows()]

sft_train_ds = Dataset.from_list(sft_train_list)
sft_eval_ds = Dataset.from_list(sft_eval_list)
dpo_train_ds = Dataset.from_list(dpo_train_list)

sft_train_ds.to_json(f"{WORK_DIR}/sft_train.jsonl", force_ascii=False)
sft_eval_ds.to_json(f"{WORK_DIR}/sft_eval.jsonl", force_ascii=False)
dpo_train_ds.to_json(f"{WORK_DIR}/dpo_train.jsonl", force_ascii=False)
print("已导出 JSONL 数据集")

# -----------------------------
# 2) 训练前校验
# -----------------------------
def valid_json(s):
    try:
        x = json.loads(s)
        return isinstance(x, dict) and all(k in x for k in ["Question", "Method", "Conclusion"])
    except:
        return False

chosen_rate = np.mean([valid_json(x["chosen"]) for x in dpo_train_list])
rejected_rate = np.mean([valid_json(x["rejected"]) for x in dpo_train_list])
print("chosen JSON合规率:", chosen_rate)
print("rejected JSON合规率:", rejected_rate)

# -----------------------------
# 3) 模型构建
# -----------------------------
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

def load_base_with_lora():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_cfg)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    return model

# -----------------------------
# 4) Smoke Test
# -----------------------------
if RUN_SMOKE_TEST:
    print(">>> Smoke Test: SFT")
    smoke_n = min(64, len(sft_train_ds))
    sft_smoke_train = sft_train_ds.select(range(smoke_n))
    sft_smoke_eval = sft_eval_ds.select(range(min(16, len(sft_eval_ds))))

    smoke_model = load_base_with_lora()

    smoke_args = TrainingArguments(
        output_dir=f"{WORK_DIR}/smoke_sft_ckpt",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=SMOKE_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=2,
        save_steps=10,
        fp16=False,
        bf16=False,
        max_grad_norm=0.0,
        optim="paged_adamw_8bit",
        report_to="none",
        remove_unused_columns=False
    )

    smoke_sft = SFTTrainer(
        model=smoke_model,
        args=smoke_args,
        train_dataset=sft_smoke_train,
        eval_dataset=sft_smoke_eval,
        processing_class=tokenizer
    )
    smoke_sft.train()
    smoke_sft.model.save_pretrained(f"{WORK_DIR}/smoke_sft_adapter")
    tokenizer.save_pretrained(f"{WORK_DIR}/smoke_sft_adapter")
    print("Smoke SFT 通过")

    print(">>> Smoke Test: DPO")
    dpo_smoke_train = dpo_train_ds.select(range(smoke_n))

    dpo_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    dpo_base.config.pad_token_id = tokenizer.pad_token_id
    dpo_base = prepare_model_for_kbit_training(dpo_base)
    dpo_model = PeftModel.from_pretrained(dpo_base, f"{WORK_DIR}/smoke_sft_adapter", is_trainable=True)
    dpo_model.gradient_checkpointing_enable()
    dpo_model.config.use_cache = False

    smoke_dpo_args = DPOConfig(
        output_dir=f"{WORK_DIR}/smoke_dpo_ckpt",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=SMOKE_STEPS,
        learning_rate=1e-5,
        logging_steps=1,
        save_steps=10,
        fp16=False,
        bf16=False,
        max_grad_norm=0.0,
        report_to="none",
        remove_unused_columns=False,
        max_length=256
    )

    smoke_dpo = DPOTrainer(
        model=dpo_model,
        ref_model=None,
        args=smoke_dpo_args,
        train_dataset=dpo_smoke_train,
        processing_class=tokenizer
    )
    smoke_dpo.train()
    print("Smoke DPO 通过")

# -----------------------------
# 5) 全量 SFT
# -----------------------------
if RUN_FULL_TRAIN:
    print(">>> 开始全量 SFT")
    sft_model = load_base_with_lora()

    sft_args = TrainingArguments(
        output_dir=f"{WORK_DIR}/sft_ckpts",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=200,
        save_total_limit=2,
        fp16=False,
        bf16=False,
        max_grad_norm=0.0,
        optim="paged_adamw_8bit",
        report_to="none",
        remove_unused_columns=False
    )

    sft_trainer = SFTTrainer(
        model=sft_model,
        args=sft_args,
        train_dataset=sft_train_ds,
        eval_dataset=sft_eval_ds,
        processing_class=tokenizer
    )
    sft_trainer.train()
    sft_trainer.model.save_pretrained(SFT_SAVE_DIR)
    tokenizer.save_pretrained(SFT_SAVE_DIR)

    pd.DataFrame(sft_trainer.state.log_history).to_csv(
        f"{WORK_DIR}/sft_log_history.csv", index=False, encoding="utf-8-sig"
    )
    print("SFT完成，保存:", SFT_SAVE_DIR)

# -----------------------------
# 6) 全量 DPO —— 你提供的极限省显存版（100% 原样保留）
# -----------------------------
import gc
import inspect

if RUN_FULL_TRAIN:
    print(">>> 开始全量 DPO（极限省显存版）")

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    gc.collect()
    torch.cuda.empty_cache()

    MAX_PROMPT_TOKENS = 192
    MAX_ANSWER_TOKENS = 96
    TRAIN_FRACTION = 0.5

    def truncate_by_tokens(text, max_tokens):
        ids = tokenizer(text, add_special_tokens=False).input_ids[:max_tokens]
        return tokenizer.decode(ids, skip_special_tokens=True)

    def shrink_example(ex):
        p = truncate_by_tokens(ex["prompt"], MAX_PROMPT_TOKENS)
        c = normalize_completion(truncate_by_tokens(ex["chosen"], MAX_ANSWER_TOKENS))
        r = normalize_completion(truncate_by_tokens(ex["rejected"], MAX_ANSWER_TOKENS))
        return {"prompt": p, "chosen": c, "rejected": r}

    dpo_small = dpo_train_ds.map(shrink_example, num_proc=1)

    if TRAIN_FRACTION < 1.0:
        n = len(dpo_small)
        k = max(64, int(n * TRAIN_FRACTION))
        rng = np.random.default_rng(SEED)
        idx = rng.choice(n, size=k, replace=False)
        dpo_small = dpo_small.select(sorted(idx.tolist()))

    print("DPO训练样本数:", len(dpo_small))

    dpo_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    dpo_base = prepare_model_for_kbit_training(dpo_base)
    dpo_base.config.pad_token_id = tokenizer.pad_token_id
    dpo_base.config.use_cache = False
    dpo_base.gradient_checkpointing_enable()

    dpo_model = PeftModel.from_pretrained(dpo_base, SFT_SAVE_DIR, is_trainable=True)

    for p in dpo_model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    cfg_kwargs = dict(
        output_dir="/kaggle/working/dpo_ckpts",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=64,
        num_train_epochs=1,
        learning_rate=5e-6,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        fp16=False,
        bf16=False,
        max_grad_norm=0.0,
        report_to="none",
        remove_unused_columns=False,
        max_length=256,
    )

    if "reference_free" in inspect.signature(DPOConfig).parameters:
        cfg_kwargs["reference_free"] = True

    dpo_args = DPOConfig(**cfg_kwargs)

    gc.collect()
    torch.cuda.empty_cache()

    dpo_trainer = DPOTrainer(
        model=dpo_model,
        ref_model=None,
        args=dpo_args,
        train_dataset=dpo_small,
        processing_class=tokenizer
    )

    dpo_trainer.train()

    dpo_trainer.model.save_pretrained(DPO_SAVE_DIR)
    tokenizer.save_pretrained(DPO_SAVE_DIR)

    pd.DataFrame(dpo_trainer.state.log_history).to_csv(
        "/kaggle/working/dpo_log_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("DPO完成，模型已保存到:", DPO_SAVE_DIR)

print("全部流程正常结束！")